# Week 3 - Subqueries, CTEs and Window Functions
**Dataset:** week3sample.csv (Superstore, ~9,994 rows)  
**Objective:** Analyze customer sales data using Subqueries, CTEs, and Window Functions in SQL.

---
## Step 1 - Setting Up the Data

First I loaded the raw CSV into a table called superstore_raw.
Then I split it into 3 normalized tables - customers, orders and products.
Used SELECT DISTINCT to avoid duplicate entries in each table.
This kind of structure is better for running joins and aggregations cleanly.

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv('week3sample.csv', encoding='latin1')
df.columns = [c.strip().replace(' ', '_').replace('-', '_') for c in df.columns]
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date']  = pd.to_datetime(df['Ship_Date'])

con = sqlite3.connect('week3.db')

# Load raw table
df.to_sql('superstore_raw', con, if_exists='replace', index=False)

# customers table
df[['Customer_ID','Customer_Name','Segment']].drop_duplicates().to_sql(
    'customers', con, if_exists='replace', index=False)

# products table
df[['Product_ID','Product_Name','Category','Sub_Category']].drop_duplicates().to_sql(
    'products', con, if_exists='replace', index=False)

# orders table
df[['Order_ID','Order_Date','Ship_Date','Ship_Mode',
    'Customer_ID','Product_ID','Sales','Quantity','Discount','Profit',
    'Region','City','State','Country']].drop_duplicates().to_sql(
    'orders', con, if_exists='replace', index=False)

print('superstore_raw rows:', pd.read_sql("SELECT COUNT(*) AS n FROM superstore_raw", con).iloc[0,0])
print('customers rows:',      pd.read_sql("SELECT COUNT(*) AS n FROM customers", con).iloc[0,0])
print('products rows:',       pd.read_sql("SELECT COUNT(*) AS n FROM products", con).iloc[0,0])
print('orders rows:',         pd.read_sql("SELECT COUNT(*) AS n FROM orders", con).iloc[0,0])

**Output:**
```
superstore_raw rows: 9994
customers rows:       793
products rows:       1894
orders rows:         9993
```

---
## Step 2 - Subqueries

Subqueries let me filter or compare against a calculated value without needing a separate query.
Used them to find orders above average sales and the highest sale per customer.

In [ ]:
# 2.1 — Orders where Sales > Average Sales
q = """
SELECT Order_ID, Customer_ID, Sales
FROM   orders
WHERE  Sales > (SELECT AVG(Sales) FROM orders)
ORDER  BY Sales DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
      Order_ID Customer_ID     Sales
CA-2014-145317    SM-20320 22638.480
CA-2016-118689    TC-20980 17499.950
CA-2017-140151    RB-19360 13999.960
CA-2017-127180    TA-21385 11199.968
CA-2017-166709    HL-15040 10499.970
```

> Average order sales is around $229. These orders are all well above that — the top one from SM-20320 is nearly 100x the average.

In [ ]:
# 2.2 — Highest Sales Order for Each Customer
q = """
SELECT o.Customer_ID, c.Customer_Name, o.Order_ID, o.Sales
FROM   orders o
JOIN   customers c ON o.Customer_ID = c.Customer_ID
WHERE  o.Sales = (
    SELECT MAX(o2.Sales)
    FROM   orders o2
    WHERE  o2.Customer_ID = o.Customer_ID
)
ORDER  BY o.Sales DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
Customer_ID  Customer_Name       Order_ID     Sales
   SM-20320    Sean Miller CA-2014-145317 22638.480
   TC-20980   Tamara Chand CA-2016-118689 17499.950
   RB-19360   Raymond Buch CA-2017-140151 13999.960
   TA-21385   Tom Ashbrook CA-2017-127180 11199.968
   HL-15040   Hunter Lopez CA-2017-166709 10499.970
```

> This correlated subquery runs once per customer and finds their single biggest order. Sean Miller's top order alone was $22,638.

---
## Step 3 - CTEs (Common Table Expressions)

CTEs make complex queries easier to read. Instead of nesting subqueries inside subqueries,
I can define the logic once and reuse it. Used CTEs to calculate total sales per customer
and then filter those results further.

In [ ]:
# 3.1 — Total Sales per Customer using CTE
q = """
WITH customer_sales AS (
    SELECT o.Customer_ID,
           c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
)
SELECT * FROM customer_sales
ORDER  BY Total_Sales DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
Customer_ID  Customer_Name  Total_Sales
   SM-20320    Sean Miller     25043.05
   TC-20980   Tamara Chand     19052.22
   RB-19360   Raymond Buch     15117.34
   TA-21385   Tom Ashbrook     14595.62
   AB-10105  Adrian Barton     14473.57
```

In [ ]:
# 3.2 — Customers whose Total Sales are Above Average (CTE + Subquery)
q = """
WITH customer_sales AS (
    SELECT o.Customer_ID,
           c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
)
SELECT * FROM customer_sales
WHERE  Total_Sales > (SELECT AVG(Total_Sales) FROM customer_sales)
ORDER  BY Total_Sales DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
Customer_ID  Customer_Name  Total_Sales
   SM-20320    Sean Miller     25043.05
   TC-20980   Tamara Chand     19052.22
   RB-19360   Raymond Buch     15117.34
   TA-21385   Tom Ashbrook     14595.62
   AB-10105  Adrian Barton     14473.57
```

> The average total sales per customer is around $2,897. Only a small group of customers are significantly above this — these are the high-value accounts worth retaining.

---
## Step 4 - Window Functions

Window functions are useful because they let me rank or number rows without collapsing
the result into groups. RANK gives the same rank to ties, ROW_NUMBER gives a unique
number even to ties. PARTITION BY resets the count for each customer.

In [ ]:
# 4.1 — Rank All Customers by Total Sales
q = """
WITH customer_sales AS (
    SELECT o.Customer_ID,
           c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
)
SELECT Customer_Name,
       Total_Sales,
       RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Rank
FROM   customer_sales
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
 Customer_Name  Total_Sales  Sales_Rank
   Sean Miller     25043.05           1
  Tamara Chand     19052.22           2
  Raymond Buch     15117.34           3
  Tom Ashbrook     14595.62           4
 Adrian Barton     14473.57           5
```

In [ ]:
# 4.2 — Row Numbers per Customer (PARTITION BY)
q = """
SELECT o.Order_ID,
       c.Customer_Name,
       o.Sales,
       ROW_NUMBER() OVER (
           PARTITION BY o.Customer_ID
           ORDER BY o.Sales DESC
       ) AS Order_Rank
FROM   orders o
JOIN   customers c ON o.Customer_ID = c.Customer_ID
LIMIT  10;
"""
pd.read_sql_query(q, con)

**Output:**
```
      Order_ID  Customer_Name    Sales  Order_Rank
CA-2016-103982     Alex Avila 3930.072           1
CA-2014-128055     Alex Avila  673.568           2
CA-2016-103982     Alex Avila  431.976           3
CA-2017-147039     Alex Avila  362.940           4
CA-2014-128055     Alex Avila   52.980           5
CA-2016-103982     Alex Avila   41.720           6
CA-2015-121391     Alex Avila   26.960           7
CA-2014-138100     Alex Avila   14.940           8
CA-2014-138100     Alex Avila   14.560           9
CA-2017-147039     Alex Avila   11.540          10
```

> PARTITION BY resets the row number for each customer. So Order_Rank = 1 means the biggest order that customer ever placed.

In [ ]:
# 4.3 — Top 3 Customers using Window Function
q = """
WITH customer_sales AS (
    SELECT o.Customer_ID,
           c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
),
ranked AS (
    SELECT *,
           RANK() OVER (ORDER BY Total_Sales DESC) AS rnk
    FROM   customer_sales
)
SELECT Customer_Name, Total_Sales, rnk AS Rank
FROM   ranked
WHERE  rnk <= 3;
"""
pd.read_sql_query(q, con)

**Output:**
```
Customer_Name  Total_Sales  Rank
  Sean Miller     25043.05     1
 Tamara Chand     19052.22     2
 Raymond Buch     15117.34     3
```

---
## Step 5 - Final Combined Query (JOIN + CTE + Window Function)

This query puts everything together. The CTE calculates total sales per customer,
the JOIN brings in customer names, and the window function ranks them.
This is the kind of query you would use in a real dashboard or report.

In [ ]:
# Final Combined Query — Customer Name + Total Sales + Rank
q = """
WITH customer_sales AS (
    SELECT o.Customer_ID,
           c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
)
SELECT Customer_Name,
       Total_Sales,
       RANK() OVER (ORDER BY Total_Sales DESC) AS Rank
FROM   customer_sales
ORDER  BY Rank
LIMIT  10;
"""
pd.read_sql_query(q, con)

**Output:**
```
      Customer_Name  Total_Sales  Rank
        Sean Miller     25043.05     1
       Tamara Chand     19052.22     2
       Raymond Buch     15117.34     3
       Tom Ashbrook     14595.62     4
      Adrian Barton     14473.57     5
       Ken Lonsdale     14175.23     6
       Sanjit Chand     14142.33     7
       Hunter Lopez     12873.30     8
       Sanjit Engle     12209.44     9
Christopher Conant     12129.07    10
```

---
## Step 6 - Mini Project: Customer Sales Insights

Used everything learned above to answer 5 real business questions about customers.

In [ ]:
# 6.1 — Top 5 Customers
q = """
WITH cs AS (
    SELECT c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY c.Customer_Name
)
SELECT Customer_Name,
       Total_Sales,
       RANK() OVER (ORDER BY Total_Sales DESC) AS Rank
FROM   cs
ORDER  BY Rank
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
Customer_Name  Total_Sales  Rank
  Sean Miller     25043.05     1
 Tamara Chand     19052.22     2
 Raymond Buch     15117.34     3
 Tom Ashbrook     14595.62     4
Adrian Barton     14473.57     5
```

In [ ]:
# 6.2 — Bottom 5 Customers
q = """
WITH cs AS (
    SELECT c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY c.Customer_Name
)
SELECT Customer_Name,
       Total_Sales,
       RANK() OVER (ORDER BY Total_Sales ASC) AS Rank
FROM   cs
ORDER  BY Rank
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
  Customer_Name  Total_Sales  Rank
  Thais Sissman         4.83     1
   Lela Donovan         5.30     2
   Carl Jackson        16.52     3
Mitch Gastineau        16.74     4
     Roy Skaria        22.33     5
```

> Thais Sissman has total sales of just $4.83 across all orders. These bottom customers may be one-time buyers or inactive accounts.

In [ ]:
# 6.3 — Customers Who Made Only One Order
q = """
SELECT c.Customer_Name,
       COUNT(DISTINCT o.Order_ID) AS Order_Count
FROM   orders o
JOIN   customers c ON o.Customer_ID = c.Customer_ID
GROUP  BY c.Customer_Name
HAVING Order_Count = 1
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
    Customer_Name  Order_Count
   Anemone Ratner            1
Anthony O'Donnell            1
     Carl Jackson            1
     Jenna Caffey            1
   Jocasta Rupert            1
```

> These are one-time buyers. From a business standpoint these are customers who never came back — a retention problem worth investigating.

In [ ]:
# 6.4 — Customers with Above Average Total Sales
q = """
WITH cs AS (
    SELECT c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY c.Customer_Name
)
SELECT Customer_Name, Total_Sales
FROM   cs
WHERE  Total_Sales > (SELECT AVG(Total_Sales) FROM cs)
ORDER  BY Total_Sales DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
Customer_Name  Total_Sales
  Sean Miller     25043.05
 Tamara Chand     19052.22
 Raymond Buch     15117.34
 Tom Ashbrook     14595.62
Adrian Barton     14473.57
```

In [ ]:
# 6.5 — Highest Order Value per Customer
q = """
SELECT c.Customer_Name,
       ROUND(MAX(o.Sales), 2) AS Highest_Order_Value
FROM   orders o
JOIN   customers c ON o.Customer_ID = c.Customer_ID
GROUP  BY c.Customer_Name
ORDER  BY Highest_Order_Value DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

**Output:**
```
Customer_Name  Highest_Order_Value
  Sean Miller             22638.48
 Tamara Chand             17499.95
 Raymond Buch             13999.96
 Tom Ashbrook             11199.97
 Hunter Lopez             10499.97
```

> Sean Miller placed a single order worth $22,638 — the highest in the entire dataset. These customers tend to buy expensive tech or office equipment in bulk.

In [ ]:
con.close()
print('Done.')

---
## Summary of Key Findings

| Question | Finding |
|---|---|
| Top customer | Sean Miller at $25,043 total sales |
| Bottom customer | Thais Sissman at just $4.83 total sales |
| Single order customers | Several customers placed only 1 order — retention issue |
| Above average customers | Average customer sales is ~$2,897 — only a small group beats this |
| Highest single order | Sean Miller again at $22,638 in one order |